## PydanticOutputParser

`PydanticOutputParser`는 언어 모델의 출력을 **구조화된 정보**로 변환하는 데 도움을 주는 클래스입니다. 이 클래스는 단순 텍스트 응답 대신 **명확하고 체계적인 형태로 필요한 정보를 제공**할 수 있습니다.

## 주요 메서드

`PydanticOutputParser` (대부분의 OutputParser에 해당)에는 주로 **두 가지 핵심 메서드**가 구현되어야 합니다.

- **`get_format_instructions()`**: 언어 모델이 출력해야 할 정보의 형식을 정의하는 지침을 제공합니다. 예를 들어, 언어 모델이 출력해야 할 데이터의 필드와 그 형태를 설명하는 지침을 문자열로 반환할 수 있습니다. 이 지침은 언어 모델이 출력을 구조화하고 특정 데이터 모델에 맞게 변환하는 데 매우 중요합니다.
- **`parse()`**: 언어 모델의 출력(문자열로 가정)을 받아 이를 특정 구조로 분석하고 변환합니다. Pydantic과 같은 도구를 사용하여 입력된 문자열을 사전 정의된 스키마에 따라 검증하고, 해당 스키마를 따르는 데이터 구조로 변환합니다.

## 참고 자료

- [Pydantic 공식 도큐먼트](https://docs.pydantic.dev/latest/)


### 01. Pydantic 출력 파서 page 163
- 출력파서OutputParser는 출력값을 구조화된 형식으로 변환.
- 답변에서 원하는 정보만을 뽑아낼때 사용.

In [7]:
# !pip --version

In [8]:
# !pip install python-dotenv

In [9]:
# !pip install -U langchain langchain-openai

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [11]:
print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH_API_KEY:", os.getenv("LANGSMITH_API_KEY")[:8]+"...")
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

OPENAI_API_KEY: sk-proj-...
LANGSMITH_API_KEY: lsv2_pt_...
LANGSMITH_PROJECT: hanwha_0902
LANGSMITH_ENDPOINT: https://api.smith.langchain.com


In [12]:
# LangSmith 추적 설정
!pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름 입력
logging.langsmith("ch6-01")

# 실시간 출력
from langchain_teddynote.messages import stream_response 


LangSmith 추적을 시작합니다.
[프로젝트명]
ch6-01


In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")



In [14]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

itertools란

Python에 기본으로 들어있는 표준 라이브러리 모듈로, 반복(iteration) 관련 도구를 모아둔 곳입니다. 설치 없이 바로 import 가능합니다. 

아래 셀에서는 [chain = prompt | llm]와 중복되므로 필요없어서 주석처리로 바꿈


from itertools import chain   # 실행됨: chain 이라는 이름에 itertools 함수를 넣음
chain = prompt | llm          # 실행됨: chain 이라는 이름에 LangChain 객체를 넣음 (이전 값 교체)

chain.stream(...)             # 이 시점의 chain = LangChain 객체

In [15]:
# from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음 이메일 내용 중 중요한 내용을 추출해 주세요. \n\n{email_conversation}"
)

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

중요한 내용 요약:

- 발신자: 김철수 (바이크코퍼레이션 상무)
- 수신자: 이은채 (Teddy International)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
- 요청 사항: ZENESIS 모델에 대한 상세 브로슈어 (기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 만남 제안
- 목적: 협력 가능성 논의 및 유통 전략, 마케팅 계획 구체화

In [16]:
print(output)

중요한 내용 요약:

- 발신자: 김철수 (바이크코퍼레이션 상무)
- 수신자: 이은채 (Teddy International)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
- 요청 사항: ZENESIS 모델에 대한 상세 브로슈어 (기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 만남 제안
- 목적: 협력 가능성 논의 및 유통 전략, 마케팅 계획 구체화


위와 같은 이메일 내용이 주어졌을 때 아래의 Pydantic 스타일로 정의된 클래스를 사용하여 이메일의 정보를 파싱해 보겠습니다.

참고로, Field 안에 `description` 은 텍스트 형태의 답변에서 주요 정보를 추출하기 위한 설명입니다. LLM 이 바로 이 설명을 보고 필요한 정보를 추출하게 됩니다. 그러므로 이 설명은 정확하고 명확해야 합니다.

In [17]:
class EmailSummary(BaseModel):
    person: str = Field(description="발신자")
    email: str = Field(description="발신자 이메일 주소")
    subject: str =Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

# PydanticOutputParser 생성
parser = PydanticOutputParser(pydantic_object=EmailSummary)
parser

PydanticOutputParser(pydantic_object=<class '__main__.EmailSummary'>)

In [18]:
# instructions 출력
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "발신자", "title": "Person", "type": "string"}, "email": {"description": "발신자 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


프롬프트를 정의합니다.

1. `question`: 유저의 질문을 받습니다.
2. `email_conversation`: 이메일 본문의 내용을 입력합니다.
3. `format`: 형식을 지정합니다.

`partial`는 항상 공통된 방식으로 가져오고 싶은 변수 가 있는 경우에 사용한다.

In [19]:
# 프롬프트 템플릿 
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# format 에 PydanticOutputParser의 부분 포맷팅(partial) 추가
prompt = prompt.partial(format=parser.get_format_instructions())
prompt

PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "발신자", "title": "Person", "type": "string"}, "email": {"description": "발신자 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person",

In [20]:
# chain 생성
chain = prompt | llm

# chain 실행하고 결과 출력
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요."
     },
)

# 결과는 JSON 형태로 출력
output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "김철수 상무가 이은채 대리에게 'ZENESIS' 자전거의 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함.",
  "date": "1월 15일 오전 10시"
}
```

마지막으로 `parser`를 사용하여 결과를 파싱하고 `EmailSummary` 객체로 변환합니다.

In [21]:
# PydanticOutputParser 를 사용하여 결과 파싱
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary="김철수 상무가 이은채 대리에게 'ZENESIS' 자전거의 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함." date='1월 15일 오전 10시'


In [22]:
structured_output

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary="김철수 상무가 이은채 대리에게 'ZENESIS' 자전거의 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함.", date='1월 15일 오전 10시')

In [23]:
structured_output.person

'김철수'

In [24]:
structured_output.email

'chulsoo.kim@bikecorporation.me'

In [25]:
structured_output.subject

'"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안'

In [26]:
# 출력 파서를 추가하여 프롬프트주입 방식을 사용하는, chain 재구성
chain = prompt | llm | parser

# chain을 실행하고 결과 출력
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 EmailSummary 객체 형태로 출력
response


EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary="김철수 상무가 이은채 대리님에게 바이크코퍼레이션의 자전거 'ZENESIS'에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 보냈습니다.", date='1월 15일 오전 10시')

### 02. with_structured_output() 바인딩 page 172

`.with_structured_output(Pydantic)`을 사용하여 출력 파서를 추가하면, 출력을 Pydantic 객체로 변환할 수 있습니다.

- 모델에 스키마를 강제하므로, 파싱오류가 거의 없음.
- 프롬프트 토근 절약.
- 속성이 여러개인 복잡한 구조체에 적합.
- 엔티티 추출, API 파라미터 매핑, RAG 구조화 데이터.

**참고**

`.with_structured_output()` 함수는 `stream()` 기능을 지원하지 않습니다.

In [ ]:
# 네이티브 API 방식 : 모델에 스키마 강제
llm_with_structered = ChatOpenAI(
    temperature=0, model="gpt-4o-mini"
).with_structured_output(EmailSummary)

answer = llm_with_structered.invoke(email_conversation)
answer

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary="김철수 상무가 이은채 대리에게 바이크코퍼레이션의 자전거 'ZENESIS' 유통 협력에 대해 논의하고자 미팅을 제안하며, 제품에 대한 상세 브로슈어 요청.", date='2024-01-08')